# Spotify Tracks Dataset - Data Cleaning and Preprocessing
This notebook loads spotify_tracks.csv, cleans invalid values, extracts Year, creates Popularity_Category, and exports spotify_cleaned.csv.

In [1]:
import os
import time
import json
import base64
from pathlib import Path
from urllib.parse import urlencode, quote
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

import numpy as np
import pandas as pd

In [2]:
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "dataset_5a"
DATA_DIR.mkdir(exist_ok=True)

candidates = [
    PROJECT_DIR / "spotify_tracks.csv",
    PROJECT_DIR / "spotify_tracks_dataset.csv",
    PROJECT_DIR / "spotify.csv",
    PROJECT_DIR / "dataset.csv",
    PROJECT_DIR.parent / "dataset.csv",
]

raw_path = next((p for p in candidates if p.exists()), None)
if raw_path is None:
    raise FileNotFoundError("No dataset found. Put spotify_tracks.csv in this folder.")
assert raw_path is not None

print(f"Using file: {raw_path}")

Using file: /home/ahmed/Downloads/DV_Project/dataset.csv


In [3]:
assert raw_path is not None
raw_file = Path(raw_path)

raw_df = pd.read_csv(raw_file, low_memory=False)
raw_df.columns = [c.strip() for c in raw_df.columns]
print("Raw shape:", raw_df.shape)
print("Columns:", list(raw_df.columns))
raw_df.head()

Raw shape: (114000, 21)
Columns: ['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [4]:
def first_existing_column(df, options):
    lookup = {c.lower(): c for c in df.columns}
    for opt in options:
        if opt.lower() in lookup:
            return lookup[opt.lower()]
    return None

def coerce_year_int(value):
    num = pd.to_numeric(pd.Series([value]), errors="coerce").iloc[0]
    if pd.isna(num):
        return None
    year = int(float(num))
    if 1900 <= year <= 2035:
        return year
    return None

def extract_year_from_release(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if not text:
        return np.nan
    parsed = pd.to_datetime(text, errors="coerce")
    if pd.notna(parsed):
        return float(parsed.year)
    if len(text) >= 4 and text[:4].isdigit():
        return float(text[:4])
    return np.nan

def get_spotify_access_token(client_id, client_secret):
    token_url = "https://accounts.spotify.com/api/token"
    raw = f"{client_id}:{client_secret}".encode("utf-8")
    auth = base64.b64encode(raw).decode("utf-8")
    body = urlencode({"grant_type": "client_credentials"}).encode("utf-8")
    req = Request(token_url, data=body, method="POST")
    req.add_header("Authorization", f"Basic {auth}")
    req.add_header("Content-Type", "application/x-www-form-urlencoded")
    with urlopen(req, timeout=30) as resp:
        payload = json.loads(resp.read().decode("utf-8"))
    return payload.get("access_token", "")

def fetch_spotify_years(track_ids, token, batch_size=50, pause_seconds=0.12):
    mapping = {}
    for i in range(0, len(track_ids), batch_size):
        chunk = track_ids[i:i + batch_size]
        url = f"https://api.spotify.com/v1/tracks?ids={','.join(chunk)}"
        req = Request(url, method="GET")
        req.add_header("Authorization", f"Bearer {token}")
        try:
            with urlopen(req, timeout=35) as resp:
                payload = json.loads(resp.read().decode("utf-8"))
        except (HTTPError, URLError):
            time.sleep(1.2)
            continue

        tracks = payload.get("tracks", [])
        for tr in tracks:
            if not tr:
                continue
            tid = tr.get("id")
            release_date = (((tr.get("album") or {}).get("release_date")) or "")
            year = coerce_year_int(extract_year_from_release(release_date))
            if tid and year is not None:
                mapping[str(tid)] = year

        time.sleep(pause_seconds)

    return mapping

def _normalize_artist_name(text):
    raw = str(text)
    primary = raw.split(';')[0].split(',')[0].split('&')[0].strip()
    return primary if primary else raw.strip()

def fetch_musicbrainz_year(track_name, artist_name):
    query = f'recording:"{track_name}" AND artist:"{artist_name}"'
    url = f"https://musicbrainz.org/ws/2/recording/?query={quote(query)}&fmt=json&limit=1"
    req = Request(url, headers={"User-Agent": "DVProjectBot/1.0 (academic-project)"})
    try:
        with urlopen(req, timeout=20) as resp:
            payload = json.loads(resp.read().decode("utf-8"))
    except (HTTPError, URLError):
        return np.nan

    recs = payload.get("recordings") or []
    if not recs:
        return np.nan
    releases = recs[0].get("releases") or []
    for rel in releases:
        y = coerce_year_int(extract_year_from_release(rel.get("date")))
        if y is not None:
            return float(y)
    return np.nan

def fetch_musicbrainz_years(df_subset, max_lookups=120):
    resolved = {}
    unique_rows = df_subset[["track_id", "track_name", "artists"]].dropna().drop_duplicates().head(max_lookups)
    total = len(unique_rows)
    for idx, row in enumerate(unique_rows.itertuples(index=False), start=1):
        tid = str(row.track_id)
        y = coerce_year_int(fetch_musicbrainz_year(str(row.track_name), _normalize_artist_name(row.artists)))
        if y is not None:
            resolved[tid] = y
        if idx % 20 == 0 or idx == total:
            print(f"MusicBrainz progress: {idx}/{total}, resolved={len(resolved)}")
        time.sleep(1.05)
    return resolved

column_aliases = {
    "track_id": ["track_id", "id"],
    "artists": ["artists", "artist", "artist_name"],
    "album_name": ["album_name", "album", "album_title"],
    "track_name": ["track_name", "title", "name"],
    "track_genre": ["track_genre", "genre"],
    "popularity": ["popularity", "track_popularity", "Popularity"],
    "duration_ms": ["duration_ms", "duration", "duration_min"],
    "danceability": ["danceability"],
    "energy": ["energy"],
    "valence": ["valence"],
    "tempo": ["tempo"],
    "loudness": ["loudness"],
    "acousticness": ["acousticness"],
    "speechiness": ["speechiness"],
    "instrumentalness": ["instrumentalness"],
    "liveness": ["liveness"],
    "release_date": ["release_date", "album_release_date", "date"],
    "year": ["year", "Year"],
}

df = pd.DataFrame()
for target, options in column_aliases.items():
    src = first_existing_column(raw_df, options)
    if src is not None:
        df[target] = raw_df[src]

print("Mapped shape:", df.shape)
print("Mapped columns:", list(df.columns))

Mapped shape: (114000, 16)
Mapped columns: ['track_id', 'artists', 'album_name', 'track_name', 'track_genre', 'popularity', 'duration_ms', 'danceability', 'energy', 'valence', 'tempo', 'loudness', 'acousticness', 'speechiness', 'instrumentalness', 'liveness']


In [5]:
numeric_cols = [
    "popularity", "duration_ms", "danceability", "energy", "valence",
    "tempo", "loudness", "acousticness", "speechiness", "instrumentalness", "liveness",
    "year",
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# 1) Primary Year source: release_date/year columns in dataset.
if "release_date" in df.columns:
    df["Year"] = df["release_date"].map(extract_year_from_release)
elif "year" in df.columns:
    df["Year"] = pd.to_numeric(df["year"], errors="coerce")
else:
    df["Year"] = np.nan

# 2) Local metadata cache source.
metadata_cache_file = DATA_DIR / "spotify_track_metadata_years.csv"
cached_map = {}
if metadata_cache_file.exists():
    cache_df = pd.read_csv(metadata_cache_file)
    if {"track_id", "Year"}.issubset(cache_df.columns):
        cache_df = cache_df.dropna(subset=["track_id", "Year"]).copy()
        cache_df["track_id"] = cache_df["track_id"].astype(str)
        cache_df["Year"] = pd.to_numeric(cache_df["Year"], errors="coerce")
        for row in cache_df.itertuples(index=False):
            y = coerce_year_int(row.Year)
            if y is not None:
                cached_map[str(row.track_id)] = y

if "track_id" in df.columns and cached_map:
    missing_mask = df["Year"].isna()
    df.loc[missing_mask, "Year"] = df.loc[missing_mask, "track_id"].astype(str).map(cached_map)

# 3) Spotify API source (best quality) when credentials are available.
client_id = os.getenv("SPOTIFY_CLIENT_ID", "").strip()
client_secret = os.getenv("SPOTIFY_CLIENT_SECRET", "").strip()
if "track_id" in df.columns and df["Year"].isna().any() and client_id and client_secret:
    unresolved_ids = df.loc[df["Year"].isna(), "track_id"].dropna().astype(str).unique().tolist()
    unresolved_ids = [tid for tid in unresolved_ids if tid and len(tid) == 22]
    if unresolved_ids:
        print(f"Spotify API enrichment for {len(unresolved_ids)} tracks...")
        try:
            token = get_spotify_access_token(client_id, client_secret)
            fetched_map = fetch_spotify_years(unresolved_ids, token)
            if fetched_map:
                df.loc[df["Year"].isna(), "Year"] = df.loc[df["Year"].isna(), "track_id"].astype(str).map(fetched_map)
                cached_map = {**cached_map, **fetched_map}
                print(f"Spotify API resolved: {len(fetched_map)}")
        except Exception as e:
            print(f"Spotify API enrichment skipped due to error: {e}")
else:
    if not (client_id and client_secret):
        print("Spotify credentials not set; skipping Spotify API enrichment.")

# 4) MusicBrainz fallback source (real metadata, slower but credential-free).
if "track_id" in df.columns and df["Year"].isna().any():
    unresolved_df = df.loc[df["Year"].isna(), ["track_id", "track_name", "artists"]].copy()
    try:
        max_lookups = int(os.getenv("MB_MAX_LOOKUPS", "120"))
    except ValueError:
        max_lookups = 120
    if max_lookups > 0 and not unresolved_df.empty:
        print(f"MusicBrainz enrichment attempts: up to {max_lookups} lookups...")
        mb_map = fetch_musicbrainz_years(unresolved_df, max_lookups=max_lookups)
        if mb_map:
            df.loc[df["Year"].isna(), "Year"] = df.loc[df["Year"].isna(), "track_id"].astype(str).map(mb_map)
            cached_map = {**cached_map, **mb_map}
            print(f"MusicBrainz resolved: {len(mb_map)}")

# Save/update cache for next runs.
if cached_map:
    pd.DataFrame({"track_id": list(cached_map.keys()), "Year": list(cached_map.values())}).to_csv(metadata_cache_file, index=False)
    print(f"Metadata cache saved: {metadata_cache_file}")

# 5) Controlled imputation from real resolved years (artist -> genre -> global).
if "artists" in df.columns and df["Year"].notna().any():
    artist_year = df.groupby("artists")["Year"].median()
    miss = df["Year"].isna()
    if miss.any():
        df.loc[miss, "Year"] = df.loc[miss, "artists"].map(artist_year)

if "track_genre" in df.columns and df["Year"].notna().any():
    genre_year = df.groupby("track_genre")["Year"].median()
    miss = df["Year"].isna()
    if miss.any():
        df.loc[miss, "Year"] = df.loc[miss, "track_genre"].map(genre_year)

# 6) Final fallback only for stubborn missing values.
if df["Year"].notna().sum() == 0:
    df["Year"] = 2020
else:
    median_year = df["Year"].dropna().median()
    fallback_year = coerce_year_int(np.clip(median_year, 2000, 2023))
    if fallback_year is None:
        fallback_year = 2020
    df["Year"] = df["Year"].fillna(fallback_year)

df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df = df[df["Year"].notna()].copy()
df["Year"] = df["Year"].round().astype(int)
df = df[df["Year"].between(1900, 2035)]

# Remove rows with nulls in important columns.
required = ["track_name", "artists", "track_genre", "popularity", "danceability", "energy", "valence", "tempo", "loudness", "acousticness", "duration_ms"]
df = df.dropna(subset=[c for c in required if c in df.columns]).copy()

# Remove fake genres.
fake_genres = {"music", "unknown", "other", "n/a", "na", "nan", "none", ""}
if "track_genre" in df.columns:
    df["track_genre"] = df["track_genre"].astype(str).str.strip()
    df = df[~df["track_genre"].str.lower().isin(fake_genres)].copy()

# Clip numeric ranges to valid Spotify-style limits.
df["popularity"] = df["popularity"].clip(0, 100)
if "danceability" in df.columns:
    df["danceability"] = df["danceability"].clip(0, 1)
if "energy" in df.columns:
    df["energy"] = df["energy"].clip(0, 1)
if "valence" in df.columns:
    df["valence"] = df["valence"].clip(0, 1)
if "acousticness" in df.columns:
    df["acousticness"] = df["acousticness"].clip(0, 1)

# Duration is usually milliseconds; convert suspiciously small values.
if "duration_ms" in df.columns:
    tiny_mask = df["duration_ms"] < 2000
    df.loc[tiny_mask, "duration_ms"] = df.loc[tiny_mask, "duration_ms"] * 1000
    df = df[df["duration_ms"].between(30000, 900000)]

df["Popularity_Category"] = pd.cut(
    df["popularity"],
    bins=[-1, 29, 69, 100],
    labels=["Low", "Medium", "High"],
)
df = df.dropna(subset=["Popularity_Category"]).copy()
df["Popularity_Category"] = pd.Categorical(df["Popularity_Category"], categories=["Low", "Medium", "High"], ordered=True)

dedup_cols = [c for c in ["track_id", "artists", "track_name", "track_genre"] if c in df.columns]
if dedup_cols:
    df = df.drop_duplicates(subset=dedup_cols, keep="first")

print("Cleaned shape:", df.shape)
print("Nulls remaining:", float(df.isna().sum().sum()))
print("Category counts:", df["Popularity_Category"].value_counts().to_dict())
print("Year range:", float(df["Year"].min()), "to", float(df["Year"].max()))
print("Unique years:", float(df["Year"].nunique()))
df.head()

Spotify credentials not set; skipping Spotify API enrichment.
MusicBrainz enrichment attempts: up to 500 lookups...


MusicBrainz progress: 20/500, resolved=4


MusicBrainz progress: 40/500, resolved=4


MusicBrainz progress: 60/500, resolved=16


MusicBrainz progress: 80/500, resolved=30


...

MusicBrainz progress: 480/500, resolved=255


MusicBrainz progress: 500/500, resolved=265


MusicBrainz resolved: 265
Metadata cache saved: /home/ahmed/Downloads/DV_Project/spotify_trends_dashboard/dataset_5a/spotify_track_metadata_years.csv


Cleaned shape: (113382, 18)
Nulls remaining: 0.0
Category counts: {'Medium': 57920, 'Low': 49994, 'High': 5468}
Year range: 1987.0 to 2026.0
Unique years: 29.0


,track_id,artists,album_name,track_name,track_genre,popularity,duration_ms,danceability,energy,valence,tempo,loudness,acousticness,speechiness,instrumentalness,liveness,Year,Popularity_Category
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,acoustic,73,230666,0.676,0.4610,0.715,87.917,-6.746,0.0322,0.1430,0.000001,0.3580,2017,High
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,acoustic,55,149610,0.420,0.1660,0.267,77.489,-17.235,0.9240,0.0763,0.000006,0.1010,2017,Medium
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,acoustic,57,210826,0.438,0.3590,0.120,76.332,-9.734,0.2100,0.0557,0.000000,0.1170,2021,Medium
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,acoustic,71,201933,0.266,0.0596,0.143,181.740,-18.515,0.9050,0.0363,0.000071,0.1320,2017,High
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,acoustic,82,198853,0.618,0.4430,0.167,119.949,-9.681,0.4690,0.0526,0.000000,0.0829,2017,High


In [6]:
root_output = PROJECT_DIR / "spotify_cleaned.csv"
data_dir_output = DATA_DIR / "spotify_cleaned.csv"

df.to_csv(root_output, index=False)
df.to_csv(data_dir_output, index=False)

print(f"Saved cleaned file: {root_output}")
print(f"Saved cleaned file: {data_dir_output}")
print("Final columns:", list(df.columns))

Saved cleaned file: /home/ahmed/Downloads/DV_Project/spotify_trends_dashboard/spotify_cleaned.csv
Saved cleaned file: /home/ahmed/Downloads/DV_Project/spotify_trends_dashboard/dataset_5a/spotify_cleaned.csv
Final columns: ['track_id', 'artists', 'album_name', 'track_name', 'track_genre', 'popularity', 'duration_ms', 'danceability', 'energy', 'valence', 'tempo', 'loudness', 'acousticness', 'speechiness', 'instrumentalness', 'liveness', 'Year', 'Popularity_Category']


## Run Dashboard
After running all cells, execute python app.py to launch the full interactive dashboard.